In [1]:
# !pip install -U datasets
# !pip install -U translators

### Dataset loading and preparation

In [2]:
from datasets import load_dataset

ds = load_dataset("facebook/xnli", "en")

m:\python_projects\AlignScore\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 392702
    })
    test: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 5010
    })
    validation: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 2490
    })
})

In [4]:
import random

SEED=2025
random.seed(SEED)

ds_subset = ds['train'].shuffle(seed=SEED).select(range(10010))

In [5]:
ds_subset

Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 10010
})

In [6]:
ds_subset[:5]

{'premise': ["yeah well that that says a lot for the for his ability though i think for Staubach 's i mean you know",
  "Normally , I simply wouldn 't invite him and would explain that it was a small ceremony ( which it is ) .",
  "and that 's what we 're going to talk about okay and uh uh okay where do you work Karen",
  "um-hum well we 're vegetarians and we just became vegetarians for i i just became a vegetarian over the past uh year and a half and it 's a real challenge to find foods proper foods to eat and it 's a real challenge not to become like what they call a junk food junkie where your menu is composed of uh things that you want to eat that aren 't vegetable you know that aren 't don 't contain meat products or don 't contain animal products but maybe aren 't you know uh balanced meal so to speak so but we we feel a lot better since we 've become vegetarians",
  'I have an irritation , not a problem , but I thought perhaps you could offer me a palliative .'],
 'hypothesis':

### Translation functions

In [23]:
import translators as ts

def translate_text(text, src='en', dest='ru'):
    try:
        return ts.translate_text(query_text=text, translator="yandex", from_language=src, to_language=dest)
    except Exception as e:
        print(f"Error translating text starting with '{text[:30]}...': {e}")
        return None

In [24]:
translate_text("i'm sorry, that my cat is yelling all the time")

'мне жаль, что мой кот все время орет'

In [25]:
def translate_text_batch(text_list, src='en', dest='ru', delimiter="@@@"):
    translations = [translate_text(text) for text in text_list]
    return translations

In [26]:
texts = [ds_subset[0]['premise'], ds_subset[0]['hypothesis']]

texts, ds_subset[0]['label']

(["yeah well that that says a lot for the for his ability though i think for Staubach 's i mean you know",
  "that doesn 't say much about his ability , though"],
 2)

In [27]:
print(translate_text_batch(texts, src='en', dest='ru'))

['да, что ж, это многое говорит о его способностях, хотя, я думаю, что для Штаубаха, я имею в виду, вы понимаете', 'однако это мало что говорит о его способностях']


### Translation loop

In [28]:
translated_premises = []
original_premises = []
translated_hypotheses = []
original_hypotheses = []
saved_labels = []
n_rows = 0

batch_size = 32
n = len(ds_subset)

In [29]:
from tqdm import tqdm

for i in tqdm(range(0, n, batch_size), desc="Translating"):
    batch = ds_subset[i:i+batch_size]
    premises = batch['premise']
    hypotheses = batch['hypothesis']
    labels = batch['label']
    
    premise_translations = translate_text_batch(premises)
    hypothesis_translations = translate_text_batch(hypotheses)
    
    for p, h, p_trans, h_trans, l in zip(premises, hypotheses, premise_translations, hypothesis_translations, labels):
        if n_rows < 10000 and p_trans is not None and h_trans is not None:
            original_premises.append(p)
            original_hypotheses.append(h)
            translated_premises.append(p_trans)
            translated_hypotheses.append(h_trans)
            saved_labels.append(l)
            n_rows += 1
        else:
            print("Skipping pair due to translation error.")

Translating: 100%|██████████| 313/313 [1:06:20<00:00, 12.72s/it]

Skipping pair due to translation error.
Skipping pair due to translation error.
Skipping pair due to translation error.
Skipping pair due to translation error.
Skipping pair due to translation error.
Skipping pair due to translation error.
Skipping pair due to translation error.
Skipping pair due to translation error.
Skipping pair due to translation error.
Skipping pair due to translation error.


### Prepare and dataset

In [30]:
assert len(translated_premises) == len(translated_hypotheses) == len(saved_labels) == len(original_premises) == len(original_hypotheses)

In [31]:
from datasets import Dataset

ru_set = Dataset.from_dict({
    "premise": translated_premises,
    "hypothesis": translated_hypotheses,
    "label": saved_labels,
})

en_set = Dataset.from_dict({
    "premise": original_premises,
    "hypothesis": original_hypotheses,
    "label": saved_labels,
})

In [32]:
ru_set

Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 10000
})

In [ ]:
test_size = val_size = 0.1

en_train_test = en_set.train_test_split(test_size=test_size, seed=SEED)
en_train = en_train_test['train']
en_test = en_train_test['test']
en_train_val = en_train.train_test_split(test_size=val_size / (1 - test_size), seed=SEED)
en_train = en_train_val['train']
en_val = en_train_val['test']

ru_train_test = ru_set.train_test_split(test_size=test_size, seed=SEED)
ru_train = ru_train_test['train']
ru_test = ru_train_test['test']
ru_train_val = ru_train.train_test_split(test_size=val_size / (1 - test_size), seed=SEED)
ru_train = ru_train_val['train']
ru_val = ru_train_val['test']

In [34]:
from datasets import DatasetDict

en_dataset = DatasetDict({
    'train': en_train,
    'validation': en_val,
    'test': en_test
})

ru_dataset = DatasetDict({
    'train': ru_train,
    'validation': ru_val,
    'test': ru_test
})

final_dataset = DatasetDict({
    'en': en_dataset,
    'ru': ru_dataset
})

final_dataset

DatasetDict({
    en: DatasetDict({
        train: Dataset({
            features: ['premise', 'hypothesis', 'label'],
            num_rows: 7999
        })
        validation: Dataset({
            features: ['premise', 'hypothesis', 'label'],
            num_rows: 1001
        })
        test: Dataset({
            features: ['premise', 'hypothesis', 'label'],
            num_rows: 1000
        })
    })
    ru: DatasetDict({
        train: Dataset({
            features: ['premise', 'hypothesis', 'label'],
            num_rows: 7999
        })
        validation: Dataset({
            features: ['premise', 'hypothesis', 'label'],
            num_rows: 1001
        })
        test: Dataset({
            features: ['premise', 'hypothesis', 'label'],
            num_rows: 1000
        })
    })
})

In [39]:
ru_dataset.push_to_hub("MilyaShams/xnli_ru_10k", "ru", private=False)

Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.08s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/MilyaShams/xnli_ru_10k/commit/bbcb5ddad42d76ba7b8268e9f4eec937c8bb3fc9', commit_message='Upload dataset', commit_description='', oid='bbcb5ddad42d76ba7b8268e9f4eec937c8bb3fc9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/MilyaShams/xnli_ru_10k', endpoint='https://huggingface.co', repo_type='dataset', repo_id='MilyaShams/xnli_ru_10k'), pr_revision=None, pr_num=None)

In [40]:
en_dataset.push_to_hub("MilyaShams/xnli_ru_10k", "en", private=False)

Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.23it/s]


CommitInfo(commit_url='https://huggingface.co/datasets/MilyaShams/xnli_ru_10k/commit/8853b5dd6a30c430f09aa220427949866a4d2d2f', commit_message='Upload dataset', commit_description='', oid='8853b5dd6a30c430f09aa220427949866a4d2d2f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/MilyaShams/xnli_ru_10k', endpoint='https://huggingface.co', repo_type='dataset', repo_id='MilyaShams/xnli_ru_10k'), pr_revision=None, pr_num=None)